# Laborator 1

## Cerințe inițiale laborator

Orice mediu de dezvoltare Python:
- PyCharm: https://www.jetbrains.com/pycharm/
- Spyder: https://www.spyder-ide.org/
- colab: https://colab.research.google.com/
- VSCode: https://code.visualstudio.com/
- etc

NumPy: https://numpy.org/install/

scikit-learn: https://scikit-learn.org/stable/install.html

CUDA (doar pentru PyTorch GPU): https://developer.nvidia.com/cuda-downloads/

PyTorch: https://pytorch.org/get-started/locally/

## Exerciții

### 1. Scalare Min-Max
Fie un vector tensor X de dimensiune necunoscută.

Scalați elementele acestuia în intervalul [0, 1].

In [9]:
import numpy as np
import torch

X = torch.from_numpy(np.random.uniform(low=-3, high=3, size=(3,3)))
print(X)
print(X.shape)   # Afisati dimensiunea tensorului X
X = (X - torch.min(X)) / (torch.max(X) - torch.min(X))
print(X)

assert torch.allclose(X.amin(), torch.tensor(0.0, dtype=X.dtype))
assert torch.allclose(X.amax(), torch.tensor(1.0, dtype=X.dtype))

tensor([[ 2.3870, -2.0710,  2.4672],
        [ 2.1815,  2.1677, -1.1727],
        [ 0.9637,  1.2356,  0.5805]], dtype=torch.float64)
torch.Size([3, 3])
tensor([[0.9823, 0.0000, 1.0000],
        [0.9371, 0.9340, 0.1979],
        [0.6687, 0.7286, 0.5843]], dtype=torch.float64)


**Hint**:

`torch.min(...)` / `tensor.min(...)`

`torch.max(...)` / `tensor.max(...)`

### 2. Standardizare — medie 0, varianță 1 (vector)
Fie un vector tensor X de dimensiune necunoscută.

Normalizați elementele acestuia astfel încât să aibe medie 0 și varianță 1.

In [10]:
X = torch.from_numpy(np.random.uniform(low=-3, high=3, size=(3, 3)))
print(X)
X = (X - torch.mean(X)) / torch.std(X)
print(X)

assert torch.allclose(X.mean(), torch.tensor(0.0, dtype=X.dtype), atol=1e-6)
assert torch.allclose(X.var(), torch.tensor(1.0, dtype=X.dtype), atol=1e-6)

tensor([[ 1.3457, -0.6517, -0.9035],
        [ 1.5675, -2.6963,  0.4048],
        [-1.2788, -1.3827,  2.5030]], dtype=torch.float64)
tensor([[ 0.8724, -0.3154, -0.4651],
        [ 1.0043, -1.5312,  0.3129],
        [-0.6883, -0.7501,  1.5606]], dtype=torch.float64)


**Hint**:

`torch.mean(...)` / `tensor.mean(...)`

`torch.std(...)` / `tensor.std(...)`

`torch.var(...)` / `tensor.var(...)`

### 3. Scalare Min-Max pe coloane (matrice)
Fie o matrice tensor X de dimensiune necunoscută.

Scalați elementele fiecărei coloane în intervalul [0, 1].

In [17]:
X = torch.from_numpy(np.random.uniform(low=-3, high=3, size=(3, 3)))
print(X)
X = (X - torch.min(X, dim=0).values) / (torch.max(X, dim=0).values - torch.min(X, dim=0).values)
print(X)

assert torch.allclose(X.amin(dim=0), torch.zeros(X.shape[1], dtype=X.dtype))
assert torch.allclose(X.amax(dim=0), torch.ones(X.shape[1], dtype=X.dtype))

tensor([[-0.8582,  2.4483, -1.9295],
        [ 2.8498,  0.7003,  1.1369],
        [-0.0714, -0.1366,  0.2927]], dtype=torch.float64)
tensor([[0.0000, 1.0000, 0.0000],
        [1.0000, 0.3238, 1.0000],
        [0.2122, 0.0000, 0.7247]], dtype=torch.float64)


**Hint**:

Similar cu np, se pot realiza reduceri parțiale (pe linie / coloană).

Similar cu np, se pot realiza operații între matrice și vector.

### 4. Standardizare pe coloane (matrice)
Fie o matrice tensor X de dimensiune necunoscută.

Normalizați elementele fiecărei coloane astfel încât să aibe medie 0 și varianță 1.

In [21]:
X = torch.from_numpy(np.random.uniform(low=-3, high=3, size=(3, 3)))
print(X)
X = (X - torch.mean(X, dim=0)) / torch.std(X, dim=0)
print(X)

assert torch.allclose(X.mean(dim=0), torch.zeros(X.shape[1], dtype=X.dtype), atol=1e-6)
assert torch.allclose(X.var(dim=0), torch.ones(X.shape[1], dtype=X.dtype), atol=1e-6)

tensor([[-2.1718,  2.9209,  1.4597],
        [-0.4459, -0.2021,  0.8661],
        [-2.1802,  1.8580,  2.3004]], dtype=torch.float64)
tensor([[-0.5731,  0.8788, -0.1142],
        [ 1.1547, -1.0881, -0.9380],
        [-0.5816,  0.2093,  1.0522]], dtype=torch.float64)


### 5. Matricea de covarianță a trăsăturilor
Fie o matrice tensor X de dimensiune necunoscută.

Fiecare linie a matricii reprezintă un exemplu și fiecare coloană o trăsătură.

* Calculați matricea de covarință a trăsăturilor K.

$$K = \frac{1}{n-1} X_c^\top X_c, \qquad X_c = X - \bar{X}$$

unde $X_c$ este matricea centrată (fiecare coloană are media 0) și $n$ este numărul de exemple (linii ale lui $X$).

* Element cu element:

$$K_{ij} = \frac{1}{n-1} \sum_{k=1}^{n} (x_{ki} - \bar{x}_i)(x_{kj} - \bar{x}_j)$$

In [38]:
X = torch.from_numpy(np.random.uniform(low=-3, high=3, size=(2, 2)))
n = X.shape[0]
X_c = X - X.mean(dim=0)
K = (X_c.T @ X_c) / (n - 1)
print(K)

assert torch.allclose(K, K.T, atol=1e-6)  # simetrică
assert torch.allclose(K, torch.cov(X.T), atol=1e-4)  # comparație cu funcția din bibliotecă

tensor([[ 3.5114e-03, -1.6345e-01],
        [-1.6345e-01,  7.6085e+00]], dtype=torch.float64)


In [43]:
X = torch.from_numpy(np.random.uniform(low=-3, high=3, size=(3, 3)))
print(X)

X_std = (X - torch.mean(X, dim=0)) / torch.std(X, dim=0)
n = X_std.shape[0]

X_stdC = X_std - X_std.mean(dim=0)
K = (X_stdC.T @ X_stdC) / (n - 1)
print(K)

tensor([[-0.8409, -1.9124,  1.9853],
        [ 0.3305, -2.7458, -2.7995],
        [ 0.3804,  2.9693, -2.1995]], dtype=torch.float64)
tensor([[ 1.0000,  0.4117, -0.9886],
        [ 0.4117,  1.0000, -0.2695],
        [-0.9886, -0.2695,  1.0000]], dtype=torch.float64)


**Hint**:

* K[i, j] = covarianța între trăsătura i și j.

* Încercați să calculați K direct ca o operație matriceală.

### 6. Descompunere în valori/vectori proprii și proiecție PCA
Fie o matrice tensor X de dimensiune necunoscută.

Fiecare linie a matricii reprezintă un exemplu și fiecare coloană o trăsătură.

1. Calculați vectorii proprii V și valorile proprii D pentru matricea de covarianță a trăsăturilor K.

2. Sortați descrescător valorile proprii. Rearanjați vectorii proprii după valorile sortate.

3. Realizați proiecția PCA pe un număr de trăsături ales.

In [52]:
X = torch.from_numpy(np.random.uniform(low=-3, high=3, size=(3, 3)))

X_centrat = X - torch.mean(X, dim=0)
n = X.shape[0]
K = (X_centrat.T @ X_centrat) / (n - 1)
print(K)

D, V = torch.linalg.eigh(K)
print(D)
print(V)

# sortare descrescătoare a valorilor proprii și rearanjarea vectorilor proprii
indices = torch.argsort(D, descending=True)
D = D[indices]
V = V[:, indices]

# verificare: V @ diag(D) @ inv(V) == K
K_rec = V @ torch.diag(D) @ torch.linalg.inv(V)
print(torch.allclose(K, K_rec, atol=1e-4))

# proiecție PCA pe primele i trăsături
i = 3
X_PCA = X_centrat @ V[:, :i]

assert torch.allclose(K, V @ torch.diag(D) @ torch.linalg.inv(V), atol=1e-4)
assert torch.all(D[:-1] >= D[1:])  # valorile proprii sunt sortate descrescător
assert torch.allclose(V.T @ V, torch.eye(V.shape[0], dtype=V.dtype), atol=1e-4)  # vectori proprii ortonormali
assert X_PCA.shape == (X.shape[0], i)

tensor([[ 6.7944,  2.6287, -1.2840],
        [ 2.6287,  2.4565, -1.3856],
        [-1.2840, -1.3856,  0.7915]], dtype=torch.float64)
tensor([7.3121e-17, 1.6037e+00, 8.4386e+00], dtype=torch.float64)
tensor([[-0.0424, -0.4882, -0.8717],
        [ 0.5249,  0.7315, -0.4352],
        [ 0.8501, -0.4760,  0.2252]], dtype=torch.float64)
True


**Hint**:

`torch.linalg.eigh(...)` / `torch.linalg.eig(...)`

`torch.argsort(...)`

Rearanjarea a fost făcută cu succes dacă V D V<sup>-1</sup> = K, unde D este matricea diagonală cu valorile proprii pe diagonală.

`torch.diag(...)`

`torch.linalg.inv(...)`

`torch.allclose(...)`

X_PCA = X_centrat · V<sub>i</sub>, unde V<sub>i</sub> reprezintă matricea vectorilor proprii cu toate liniile si primele i coloane (cele mai importante i trăsături).

### 7. Proiecție PCA albită (whitening)
Fie o matrice tensor X de dimensiune necunoscută.

Fiecare linie a matricii reprezintă un exemplu și fiecare coloană o trăsătură.

Calculați proiecția PCA albită pe un număr de trăsături.

In [63]:
X = torch.from_numpy(np.random.uniform(low=-3, high=3, size=(3, 3)))
X_c = X - torch.mean(X, dim=0)
n = X.shape[0]
K = (X_c.T @ X_c) / (n - 1)
D, V = torch.linalg.eigh(K)

idx = torch.argsort(D, descending=True)
D = D[idx]
V = V[:, idx]

i = 2
X_PCA_white = X_c @ V[:, :i] / torch.sqrt(D[:i])


# covarianța trăsăturilor albite trebuie să fie ~identitate
K_white = (X_PCA_white.T @ X_PCA_white) / (X_PCA_white.shape[0] - 1)
assert torch.allclose(K_white, torch.eye(i, dtype=K_white.dtype), atol=1e-4)

**Hint**:

X_PCA_white = X_centrat · V<sub>i</sub> · D<sub>i</sub><sup>-0.5</sup>, unde D<sub>i</sub> este matricea pătratică cu primele i valori proprii pe diagonală.

### 8. Clasificare Bayes pe setul de date Iris (PCA și PCA albit)
Considerați setul de date iris, cu 4 trăsături și 3 clase, 50 de exemple per clasă.

1. Normalizați fiecare trăsătură la medie 0 și varianță 1.

2. Calculați matricea de covarianță K pentru trăsături.

3. Calculați vectorii proprii V și valorile proprii D pentru K.

4. Pentru cele mai importante 4, 3, 2, 1 trăsături realizați proiecția PCA și PCA albită.

5. Împărțiti setul de date în antrenare (40 de exemple pentru fiecare clasă) și testare (restul de 10 exemple pentru fiecare clasă). Puteți realiza împărțirea și la început cât timp folosiți toate datele pentru preprocesare.

6. Realizați o clasificare Bayes. Presupuneți toate trăsăturile ca având distribuție normală. Calculați acuratețea clasificării.

In [77]:
import torch
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X = torch.tensor(iris.data, dtype=torch.float32)
y = torch.tensor(iris.target)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

**Hint**:

`train_test_split(...)` — apelați-l separat pentru fiecare clasă, apoi concatenați rezultatele cu `torch.cat`.

Pentru stabilitate numerică, calculați probabilitățile sub logaritm.

`torch.sum(...)` / `torch.log(...)` / `torch.argmax(...)`

`torch.linalg.inv(...)` / `torch.logdet(...)`

Media pe clasă: `X_train[y_train == c].mean(dim=0)`.

Prior-ul unei clase: proporția exemplelor din `X_train` care aparțin acelei clase — `(y_train == c).float().mean()`.

In [78]:
# Standardizați X_train și X_test la medie 0 / varianță 1 (ca la ex. 4).
# Important: media și deviația standard se calculează DOAR pe X_train,
# apoi se aplică (aceleași valori) și pe X_test — altfel "vedeți" datele
# de test încă din faza de antrenare (data leakage).
X_mean, X_std = torch.mean(X_train, dim=0), torch.std(X_train, dim=0)
X_train = (X_train - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

assert torch.allclose(X_train.mean(dim=0), torch.zeros(4), atol=1e-6)
assert torch.allclose(X_train.var(dim=0), torch.ones(4), atol=1e-6)

In [81]:
# Calculați matricea de covarianță K pe X_train (ca la ex. 5) și descompunerea
# proprie (D, V), cu valorile proprii sortate descrescător (ca la ex. 6).
X_train_C = X_train - torch.mean(X_train, dim=0)
n = X_train.shape[0]
K = (X_train_C.T @ X_train_C) / (n - 1)
D, V = torch.linalg.eigh(K)

indices = torch.argsort(D, descending=True)
D = D[indices]
V = V[:, indices]

print("Valori proprii:", D)
assert torch.all(D[:-1] >= D[1:])

Valori proprii: tensor([2.9071, 0.9227, 0.1513, 0.0190])


In [82]:
import math
import torch

def bayes_gaussian_classify(X_train, y_train, X_test):
    classes = torch.unique(y_train)          # tensor([0, 1, 2])
    n_features = X_train.shape[1]
    n_train = X_train.shape[0]

    # TODO 1: dicționar {clasă: vectorul de medii}
    mus = {}
    for c in classes:
        c = c.item()                              # tensor(0) -> 0 (cheie simplă)
        mus[c] = X_train[y_train == c].mean(dim=0)    # shape (d,)

    # TODO 2: covarianța comună (pooled) + inversă + log-determinant
    X_centrat = X_train - X_train.mean(dim=0, keepdim=True)
    K_hat = (X_centrat.T @ X_centrat) / (n_train - 1)
    K_hat = K_hat + 1e-6 * torch.eye(n_features, dtype=K_hat.dtype)
    K_inv = torch.linalg.inv(K_hat)
    K_logdet = torch.logdet(K_hat)

    # TODO 3: dicționar {clasă: prior}
    priors = {}
    for c in classes:
        c = c.item()
        priors[c] = (y_train == c).float().mean()

    # TODO 4-6: dicționar {clasă: log-posterior pentru toate exemplele de test}
    log_posteriors = {}
    for c in classes:
        c = c.item()
        diff = X_test - mus[c]                                    # (m, d)
        maha = ((diff @ K_inv) * diff).sum(dim=1)                 # (m,)
        log_likelihood = -0.5 * maha - 0.5 * (n_features * math.log(2 * math.pi) + K_logdet)
        log_posteriors[c] = log_likelihood + torch.log(priors[c])  # (m,)

    # transformăm dicționarul într-o matrice (m, C), în ordinea claselor
    scores = torch.stack([log_posteriors[c.item()] for c in classes], dim=1)

    return classes[scores.argmax(dim=1)]

In [83]:
# Pentru fiecare număr de trăsături păstrate (1, 2, 3, 4), realizați proiecția
# PCA și proiecția PCA albită (ca la ex. 6 și 7) pe X_train / X_test, apoi rulați
# clasificatorul Bayes definit mai sus și calculați acuratețea pe X_test.
for n_feat in range(1, X.shape[1] + 1):
    Vi, Di = V[:, :n_feat], D[:n_feat]

    X_train_pca = X_train @ Vi
    X_test_pca = X_test @ Vi
    X_train_white = X_train_pca / torch.sqrt(Di)
    X_test_white = X_test_pca / torch.sqrt(Di)

    for nume, Xtr, Xte in [("PCA", X_train_pca, X_test_pca),
                           ("PCA albit", X_train_white, X_test_white)]:
        y_pred = bayes_gaussian_classify(Xtr, y_train, Xte)
        assert y_pred.shape == y_test.shape
        acc = (y_pred == y_test).float().mean().item()
        print(f"n_features={n_feat} [{nume}]: acuratețe = {acc:.4f}")

n_features=1 [PCA]: acuratețe = 0.9000
n_features=1 [PCA albit]: acuratețe = 0.9000
n_features=2 [PCA]: acuratețe = 0.8000
n_features=2 [PCA albit]: acuratețe = 0.8000
n_features=3 [PCA]: acuratețe = 0.8000
n_features=3 [PCA albit]: acuratețe = 0.8000
n_features=4 [PCA]: acuratețe = 0.8333
n_features=4 [PCA albit]: acuratețe = 0.8333


**Hint**:

`train_test_split(...)`

Pentru stabilitate numerică, calculați probabilitățile sub logaritm.

`torch.sum(...)` / `torch.prod(...)`

`torch.log(...)`

`torch.argmax(...)`